User behavior and topic cluster
-------------------------------

We check various aspects and characteristics of the data such as the relative contribution
of different topics/subreddits to reddit traffic and overall number of comments.
Total number of comments in May 2015 is

In [ ]:
library(dplyr)
library(ggplot2)
library(RColorBrewer)
library(lubridate)
db <- src_sqlite('../input/database.sqlite', create = F)
db_subset <- db %>%
                tbl('May2015')
all_posts <- db_subset %>% 
                select(id) %>% 
                    collect() %>%
                        data.frame() %>%
                            nrow()
print(all_posts)

Which subreddits contributed to most number of comments?

In [ ]:
db_subreddit <- db_subset %>%
                select(subreddit) %>%
                    collect() %>%
                        data.frame() %>%
                          group_by(subreddit) %>%
                            summarise(percent_posts = n()/all_posts*100)%>%
                              arrange(desc(percent_posts))

print(slice(db_subreddit,1:5))

We look at the distribution of relative contribution of posts

In [ ]:
w_posts_vec <- collect(db_subreddit) %>% .$percent_posts 
q_posts <- w_posts_vec %>% quantile()
print(q_posts)
relative_w_group <- tapply(w_posts_vec, cut(w_posts_vec, breaks=q_posts[3:5]), FUN=sum)
names(relative_w_group) <- c("Q1-Q3","Q4")
print(relative_w_group["Q4"])

Q4 has more than 99% of the posts - top 25% of subreddits contribute to more than 99% of
comments. 

What about users? If we ask the same question?

In [ ]:
db_user <- db_subset %>%
            filter(author != "[deleted]") %>%
                select(author) %>%
                    collect() %>%
                        data.frame() %>%
                          group_by(author) %>%
                            summarise(percent_posts = n()/all_posts*100)%>%
                              arrange(desc(percent_posts))

w_user_vec <- db_user %>% .$percent_posts 
q_users <- w_user_vec %>% quantile()

print(q_users)

relative_w_3quser_group <- tapply(w_user_vec, cut(w_user_vec, breaks=q_users[3:5]), FUN=sum)
names(relative_w_3quser_group) <- c("Q1-Q3","Q4")
print(relative_w_3quser_group["Q4"])

This means top- 25% of users are responsible to generate 80% of the comments
as expected the distribution of users in less skewed. 

Hourly activity of redditors
----------------------------

Now we check the time for number of posts - what is the range of time redditors are most active?


In [ ]:
total_posts <- db_subset %>% select(id)%>% collect() %>% data.frame() %>% nrow()

We check the percent post roughly, that is we collect average number of posts 
that hour and then divide by average number of posts per day for that month

What is the average number of posts per day on May2015?

In [ ]:
avg_post_day <- db_subset %>% 
                  select(created_utc)  %>%
                    collect() %>%
                      data.frame() %>%
                        mutate(date = as.Date(as.POSIXct(created_utc,origin = '1970-01-01'))) %>%
                        select(date) %>%
                          group_by(date) %>%
                            summarise(n_posts = n())%>% 
                              .$n_posts %>%
                                mean()
print(avg_post_day)

In [ ]:
db_post_hour <- db_subset %>% 
                  select(created_utc) %>%
                    collect() %>%
                      data.frame() %>%
                        mutate(hour = hour(as.POSIXct(created_utc,origin = '1970-01-01'))) %>%
                          group_by(hour) %>%
                            summarise(percent_posts = n()/31/avg_post_day*100)

Now we visualize post per hour

In [ ]:
ggplot(data = db_post_hour, aes(x=hour, y=percent_posts)) + 
  geom_bar(stat="identity", fill = "grey50") + 
  labs(x = "hour", 
       y = "average percent of posts", 
       title = "Average percent of posts per hour")